In [3]:
# 0. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
!pip install requests beautifulsoup4 pandas tqdm --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from tqdm.notebook import tqdm
import os
import zipfile
from google.colab import files, auth
import time
import random
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

In [4]:
# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
AÑO = 2026

MES_INICIO = 1      # Mes desde el cual arranca el bucle automático
MES_FIN    = None   # None = hasta el último mes disponible del año
                    # (o un número, p. ej. 6, para detenerse antes)

DESCARGA_FINAL = "ninguna"   # "zip"        -> un solo .zip con todos los CSV generados
                         # "individual" -> descarga cada CSV por separado
                         # "ninguna"    -> solo se guardan en la carpeta de Colab

#   Ejemplo: AÑO = 2025, MES_INICIO = 1  ->  procesa Mes 01, 02, ... 12 en cadena.
#   Para otro año: cambia AÑO (y MES_INICIO si hace falta) y vuelve a ejecutar.

# ── Determinar meses válidos según el año ────────────────────────────────────
hoy        = date.today()
año_actual = hoy.year
mes_actual = hoy.month

if AÑO < año_actual:
    meses_validos = list(range(1, 13))
elif AÑO == año_actual:
    if mes_actual == 1:
        raise ValueError("No hay meses cerrados aún en este año.")
    meses_validos = list(range(1, mes_actual))      # solo meses ya cerrados
else:
    raise ValueError("Año inválido: no puede ser futuro.")

ultimo_valido = meses_validos[-1]
if MES_FIN is None:
    MES_FIN = ultimo_valido

if not (1 <= MES_INICIO <= MES_FIN <= ultimo_valido):
    raise ValueError(
        f"Rango de meses inválido ({MES_INICIO} a {MES_FIN}). "
        f"Meses disponibles para {AÑO}: 1 a {ultimo_valido}."
    )

meses = [str(m) for m in range(MES_INICIO, MES_FIN + 1)]
print(f"Año {AÑO} -> meses a procesar en cadena: {', '.join(meses)}")

# ── Carpeta de salida ─────────────────────────────────────────────────────────
ruta_guardado = "/content/Inversion_Funcion.Municipalidad"
os.makedirs(ruta_guardado, exist_ok=True)

# ── Catálogos ─────────────────────────────────────────────────────────────────
# Solo LORETO, SAN MARTÍN y UCAYALI
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTÍN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]

funciones = [
    {"codigo": "01", "funcion": "01: LEGISLATIVA"},
    {"codigo": "02", "funcion": "02: RELACIONES EXTERIORES"},
    {"codigo": "03", "funcion": "03: PLANEAMIENTO, GESTION Y RESERVA DE CONTINGENCIA"},
    {"codigo": "04", "funcion": "04: DEFENSA Y SEGURIDAD NACIONAL"},
    {"codigo": "05", "funcion": "05: ORDEN PUBLICO Y SEGURIDAD"},
    {"codigo": "06", "funcion": "06: JUSTICIA"},
    {"codigo": "07", "funcion": "07: TRABAJO"},
    {"codigo": "08", "funcion": "08: COMERCIO"},
    {"codigo": "09", "funcion": "09: TURISMO"},
    {"codigo": "10", "funcion": "10: AGROPECUARIA"},
    {"codigo": "11", "funcion": "11: PESCA"},
    {"codigo": "12", "funcion": "12: ENERGIA"},
    {"codigo": "13", "funcion": "13: MINERIA"},
    {"codigo": "14", "funcion": "14: INDUSTRIA"},
    {"codigo": "15", "funcion": "15: TRANSPORTE"},
    {"codigo": "16", "funcion": "16: COMUNICACIONES"},
    {"codigo": "17", "funcion": "17: AMBIENTE"},
    {"codigo": "18", "funcion": "18: SANEAMIENTO"},
    {"codigo": "19", "funcion": "19: VIVIENDA Y DESARROLLO URBANO"},
    {"codigo": "20", "funcion": "20: SALUD"},
    {"codigo": "21", "funcion": "21: CULTURA Y DEPORTE"},
    {"codigo": "22", "funcion": "22: EDUCACION"},
    {"codigo": "23", "funcion": "23: PROTECCION SOCIAL"},
    {"codigo": "24", "funcion": "24: PREVISION SOCIAL"},
    {"codigo": "25", "funcion": "25: DEUDA PUBLICA"},
]

nombres_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
                 "Julio","Agosto","Setiembre","Octubre","Noviembre","Diciembre"]

meses_map   = {str(i): f"{i}: {nombres_meses[i-1]}" for i in range(1, 13)}
deptos_map  = {d["codigo"]: d["departamento"] for d in departamentos}
funcion_map = {f["codigo"]: f["funcion"]      for f in funciones}

# ── Nombre del archivo de salida ──────────────────────────────────────────────
SUFIJO_REGIONES = "LOR - SMN - UCA"

def etiqueta_meses(meses):
    """Un mes -> 'Mes 01'. Varios contiguos -> 'Meses 01-03'.
    Varios no contiguos -> 'Meses 01,03,05'."""
    nums = sorted(int(m) for m in meses)
    if len(nums) == 1:
        return f"Mes {nums[0]:02d}"
    if nums == list(range(nums[0], nums[-1] + 1)):
        return f"Meses {nums[0]:02d}-{nums[-1]:02d}"
    return "Meses " + ",".join(f"{m:02d}" for m in nums)

def nombre_archivo_salida(año, meses):
    return (f"DEV Funcion.Municipalidad {año} - "
            f"{etiqueta_meses(meses)} - {SUFIJO_REGIONES}.csv")

# ── 1b. GOOGLE DRIVE ─────────────────────────────────────────────────────────
# Carpeta destino: https://drive.google.com/drive/folders/<ID>
DRIVE_FOLDER_ID = "172c-mECT7TlyKTuTyR4-wWZ5-muXoBDG"
SUBIR_A_DRIVE   = True     # False -> solo se guarda en /content (sin subir a Drive)

drive_subidos  = []        # nombres de archivos subidos correctamente
drive_fallidos = []        # (nombre, motivo) de los que no se pudieron subir

STATUS_REINTENTABLES = {403, 429, 500, 502, 503, 504}

def conectar_drive(folder_id):
    """Autentica con la cuenta de Google del usuario y valida la carpeta destino
    ANTES de iniciar el raspado, para fallar rápido si no hay acceso."""
    auth.authenticate_user()                       # ventana de login de Google
    servicio = build("drive", "v3", cache_discovery=False)
    try:
        carpeta = servicio.files().get(
            fileId=folder_id,
            fields="id,name,mimeType,capabilities(canAddChildren)",
            supportsAllDrives=True,
        ).execute()
    except HttpError as e:
        raise PermissionError(
            "No se pudo acceder a la carpeta de Drive. Verifica que la cuenta con la "
            "que iniciaste sesión tenga acceso de EDITOR a esa carpeta "
            f"(HTTP {e.resp.status})."
        ) from e

    if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError("El ID configurado no corresponde a una carpeta de Drive.")
    if not carpeta.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(
            f"Tu cuenta no puede escribir en la carpeta '{carpeta['name']}' "
            "(se requiere permiso de Editor)."
        )
    print(f"☁️  Drive conectado -> carpeta: '{carpeta['name']}'\n")
    return servicio

def subir_a_drive(servicio, ruta_local, folder_id, max_reintentos=5):
    """Sube (o actualiza si ya existe con el mismo nombre) un archivo a la carpeta
    de Drive. Subida resumible + reintentos con backoff exponencial.
    Devuelve True/False y registra el resultado; nunca lanza excepción."""
    nombre = os.path.basename(ruta_local)
    mime   = "application/zip" if nombre.lower().endswith(".zip") else "text/csv"
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")   # escape para la query

    ultimo_error = ""
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(
                q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                fields="files(id)", supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute().get("files", [])

            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:      # mismo nombre -> se sobrescribe (evita duplicados al re-ejecutar)
                servicio.files().update(
                    fileId=existentes[0]["id"], media_body=media,
                    fields="id", supportsAllDrives=True,
                ).execute()
                accion = "actualizado"
            else:
                servicio.files().create(
                    body={"name": nombre, "parents": [folder_id]},
                    media_body=media, fields="id", supportsAllDrives=True,
                ).execute()
                accion = "creado"

            drive_subidos.append(nombre)
            print(f"   ☁️ Drive ({accion}): {nombre}")
            return True

        except HttpError as e:
            ultimo_error = f"HTTP {e.resp.status}"
            if e.resp.status not in STATUS_REINTENTABLES:
                break                                   # error no recuperable (permisos, 404...)
        except Exception as e:                          # cortes de red, timeouts, etc.
            ultimo_error = str(e)

        if intento < max_reintentos:
            espera = 2 ** intento + random.random()
            print(f"   ↻ Reintento {intento}/{max_reintentos - 1} en {espera:.1f}s ({ultimo_error})")
            time.sleep(espera)

    drive_fallidos.append((nombre, ultimo_error))
    print(f"   ❌ No se pudo subir a Drive: {nombre} ({ultimo_error}). "
          f"Sigue disponible en {ruta_local}")
    return False

# Se conecta ahora (antes de raspar) para detectar problemas de acceso desde el inicio
drive = conectar_drive(DRIVE_FOLDER_ID) if SUBIR_A_DRIVE else None

# ── 2. FUNCIÓN DE EXTRACCIÓN ──────────────────────────────────────────────────
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"

def extraer_datos(anio, mes, depto, funcion):
    params = {
        "0": "", "24": "6-2-6", "23": mes, "1": "M", "37": "M",
        "5": depto, "21": depto, "8": funcion, "7": "", "y": anio
    }
    try:
        respuesta = requests.post(URL_BASE, params=params, timeout=20)
        if respuesta.status_code != 200:
            return [], (anio, mes, depto, funcion, f"HTTP {respuesta.status_code}")
        pagina = BeautifulSoup(respuesta.content, "html.parser")
    except Exception as e:
        return [], (anio, mes, depto, funcion, str(e))

    tabla = pagina.find("table", class_="Data")
    if not tabla:
        return [], None   # Sin datos no es error

    filas      = tabla.find_all("tr")
    resultados = []

    for fila in filas:
        if fila.find("th"):
            continue
        celdas = fila.find_all("td")
        if len(celdas) < 8:
            continue
        if any(td.get("colspan") for td in celdas):
            continue

        mun = " ".join(celdas[1].get_text(strip=True).split())
        dev = " ".join(celdas[7].get_text(strip=True).split())

        if not mun:
            continue
        if dev in ("", "-", " "):
            dev = "0"

        resultados.append({
            "AÑO":           anio,
            "MES":           meses_map.get(mes, mes),
            "DEPARTAMENTO":  deptos_map.get(depto, depto),
            "FUNCION":       funcion_map.get(funcion, funcion),
            "MUNICIPALIDAD": mun,
            "DEVENGADO":     dev,
        })

    return resultados, None

# ── 3. EXTRACCIÓN DE UN MES (en paralelo) ─────────────────────────────────────
def construir_tareas(mes):
    return [
        (AÑO, mes, d["codigo"], f["codigo"])
        for d in departamentos
        for f in funciones
    ]

def ejecutar_extraccion_mes(mes, max_workers=8):
    tareas  = construir_tareas(mes)
    todos   = []
    errores = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = [executor.submit(extraer_datos, *t) for t in tareas]
        with tqdm(total=len(tareas), desc=meses_map[mes], leave=False) as barra:
            for futuro in as_completed(futuros):
                registros, error = futuro.result()
                if registros:
                    todos.extend(registros)
                if error:
                    errores.append(error)
                barra.update(1)

    return todos, errores

# ── 4. BUCLE MES A MES ────────────────────────────────────────────────────────
COLUMNAS = ["AÑO", "MES", "DEPARTAMENTO", "FUNCION", "MUNICIPALIDAD", "DEVENGADO"]

resumen_meses      = []   # una fila por mes procesado
errores_totales    = []   # detalle de peticiones fallidas (todos los meses)
archivos_generados = []   # rutas de los CSV guardados

print(f"Iniciando extracción {AÑO}: {len(meses)} mes(es) x "
      f"{len(departamentos) * len(funciones)} combinaciones cada uno.\n")

for mes in meses:
    etiqueta = meses_map[mes]
    print(f"▶ {AÑO} - {etiqueta}")

    # Un fallo inesperado en un mes no debe frenar los meses siguientes
    try:
        registros, errores = ejecutar_extraccion_mes(mes, max_workers=8)
    except Exception as e:
        print(f"   ❌ Falló el mes completo: {e}\n")
        resumen_meses.append({"MES": etiqueta, "FILAS": 0,
                              "ERRORES": "mes completo", "ESTADO": "FALLÓ"})
        continue

    df_mes = pd.DataFrame(registros, columns=COLUMNAS)

    for a, m, d, f, msg in errores:
        errores_totales.append({
            "AÑO": a, "MES": meses_map.get(m, m),
            "DEPARTAMENTO": deptos_map.get(d, d),
            "FUNCION": funcion_map.get(f, f), "ERROR": msg,
        })

    if df_mes.empty:
        estado = "SIN DATOS"
        print("   ⚠️ Sin datos para este mes: no se genera archivo.\n")
    else:
        archivo = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, [mes]))
        df_mes.to_csv(archivo, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo, DRIVE_FOLDER_ID)
        estado = "OK" if not errores else "INCOMPLETO (con errores)"
        print(f"   ✅ {len(df_mes):,} filas -> {os.path.basename(archivo)}")
        if errores:
            print(f"   ⚠️ {len(errores)} consulta(s) fallaron: el archivo puede estar incompleto.")
        print()

    resumen_meses.append({"MES": etiqueta, "FILAS": len(df_mes),
                          "ERRORES": len(errores), "ESTADO": estado})

# ── 5. RESUMEN FINAL ──────────────────────────────────────────────────────────
print("═" * 70)
print(f"RESUMEN {AÑO}")
display(pd.DataFrame(resumen_meses))

if errores_totales:
    print(f"\n⚠️ Consultas con error: {len(errores_totales)}")
    display(pd.DataFrame(errores_totales))
else:
    print("\n✅ Sin errores en ningún mes.")

# ── 5b. ERRORES Y RESUMEN DE DRIVE ────────────────────────────────────────────
if SUBIR_A_DRIVE:
    if errores_totales:     # el log de consultas fallidas también se guarda en Drive
        archivo_err = os.path.join(
            ruta_guardado,
            f"ERRORES Funcion.Municipalidad {AÑO} - {etiqueta_meses(meses)}.csv")
        pd.DataFrame(errores_totales).to_csv(archivo_err, index=False, encoding="utf-8-sig")
        subir_a_drive(drive, archivo_err, DRIVE_FOLDER_ID)

    print("\n" + "═" * 70)
    print(f"☁️ DRIVE: {len(drive_subidos)} archivo(s) subido(s), {len(drive_fallidos)} fallido(s)")
    print(f"   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    for nombre, motivo in drive_fallidos:
        print(f"   ⚠️ {nombre}: {motivo}")

# ── 6. DESCARGA ───────────────────────────────────────────────────────────────
if not archivos_generados:
    print("\n⚠️ No hay archivos para descargar.")
elif DESCARGA_FINAL == "zip":
    nombre_zip = nombre_archivo_salida(AÑO, meses).replace(".csv", ".zip")
    ruta_zip   = os.path.join(ruta_guardado, nombre_zip)
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for a in archivos_generados:
            z.write(a, arcname=os.path.basename(a))
    print(f"\n📦 {len(archivos_generados)} archivo(s) comprimidos en: {nombre_zip}")
    files.download(ruta_zip)
elif DESCARGA_FINAL == "individual":
    for a in archivos_generados:
        files.download(a)   # el navegador puede pedir permiso para descargas múltiples
else:
    print(f"\n💾 Archivos guardados en {ruta_guardado}")


Año 2026 -> meses a procesar en cadena: 1, 2, 3, 4, 5, 6, 7, 8, 9


☁️  Drive conectado -> carpeta: 'DEV'

Iniciando extracción 2026: 9 mes(es) x 75 combinaciones cada uno.

▶ 2026 - 1: Enero


1: Enero:   0%|          | 0/75 [00:00<?, ?it/s]

   ☁️ Drive (creado): DEV Funcion.Municipalidad 2026 - Mes 01 - LOR - SMN - UCA.csv
   ✅ 337 filas -> DEV Funcion.Municipalidad 2026 - Mes 01 - LOR - SMN - UCA.csv

▶ 2026 - 2: Febrero


2: Febrero:   0%|          | 0/75 [00:00<?, ?it/s]

...

   ☁️ Drive (creado): DEV Funcion.Municipalidad 2026 - Mes 08 - LOR - SMN - UCA.csv
   ✅ 664 filas -> DEV Funcion.Municipalidad 2026 - Mes 08 - LOR - SMN - UCA.csv

▶ 2026 - 9: Setiembre


9: Setiembre:   0%|          | 0/75 [00:00<?, ?it/s]

   ☁️ Drive (creado): DEV Funcion.Municipalidad 2026 - Mes 09 - LOR - SMN - UCA.csv
   ✅ 675 filas -> DEV Funcion.Municipalidad 2026 - Mes 09 - LOR - SMN - UCA.csv

══════════════════════════════════════════════════════════════════════
RESUMEN 2026


,MES,FILAS,ERRORES,ESTADO
0,1: Enero,337,0,OK
1,2: Febrero,508,0,OK
2,3: Marzo,638,0,OK
3,4: Abril,628,0,OK
4,5: Mayo,664,0,OK
5,6: Junio,711,0,OK
6,7: Julio,697,0,OK
7,8: Agosto,664,0,OK
8,9: Setiembre,675,0,OK



✅ Sin errores en ningún mes.

══════════════════════════════════════════════════════════════════════
☁️ DRIVE: 9 archivo(s) subido(s), 0 fallido(s)
   https://drive.google.com/drive/folders/172c-mECT7TlyKTuTyR4-wWZ5-muXoBDG

💾 Archivos guardados en /content/Inversion_Funcion.Municipalidad
